In [1]:
import trueq as tq
import trueq.math as tqm
from config import *

t_gate_rotation = 0.180213
h_gate_rotation = 0.020135
cnot_gate_rotation = 0.073675

circs = [tq.Circuit({0: i}) for i in easy_gates]

strength = [1,2,3,4]

def process_fidelity(circuit: tq.Circuit, simulator):
    ideal_unitary_matrix = tq.Simulator().operator(circuit= circuit).upgrade().mat()
    ideal_unitary_superop = tqm.Superop.from_rowstack(ideal_unitary_matrix) 
        
    noisy_unitary_matrix = simulator.operator(circuit= circuit).upgrade().mat()
    noisy_unitary_superop = tqm.Superop.from_rowstack(noisy_unitary_matrix) 


    proc_fid = (ideal_unitary_superop.adj @ noisy_unitary_superop).fidelity 
    return proc_fid
################ Gate independent noise model #####################

angle_gate_ind = 1.81215
gate_ind_angle =  [angle_gate_ind*0.5, angle_gate_ind, angle_gate_ind*1.5, angle_gate_ind*2]


def gate_independent_simulators():

    gate_ind_simulators = {}
    for i in range(len(gate_ind_angle)):
        # Create a fresh simulator for each iteration
        sim = tq.Simulator()
        
        easy_dict = {gate: tq.Gate.rp("X", gate_ind_angle[i])@ gate.mat for gate in clifford_gates}

        def gate_replace(gate):
            return easy_dict[gate]
        
        gate_ind_simulators[strength[i]] = sim.add_gate_replace(gate_replace, match=match_clifford).add_overrotation(single_sys=t_gate_rotation, match= match_t).add_overrotation(single_sys=h_gate_rotation, match= match_h).add_overrotation(multi_sys=cnot_gate_rotation, match= match_cnot)
        print(f"Gate independent noise model infidelity with strength {strength[i]}:")
        print(1 - sum([process_fidelity(circ, gate_ind_simulators[strength[i]]) for circ in circs]) / len(circs))

    return gate_ind_simulators

gate_independent_simulators()

Gate independent noise model infidelity with strength 1:
6.251928297185128e-05
Gate independent noise model infidelity with strength 2:
0.0002500614972442694
Gate independent noise model infidelity with strength 3:
0.0005625797427984969
Gate independent noise model infidelity with strength 4:
0.0009999958659678843


{1: <trueq.simulation.simulator.Simulator at 0x1889c01d0>,
 2: <trueq.simulation.simulator.Simulator at 0x184a47770>,
 3: <trueq.simulation.simulator.Simulator at 0x1893ee810>,
 4: <trueq.simulation.simulator.Simulator at 0x18938f170>}

In [46]:

overrotation_strength = 0.0121405

gate_dep_strengths =  [overrotation_strength*0.5, overrotation_strength, overrotation_strength*1.5, overrotation_strength*2]


def gate_dependent_simulators():

    gate_dep_simulators = {}
    for i in range(len(gate_dep_strengths)):
        # Create a fresh simulator for each iteration
        sim = tq.Simulator()
        
        gate_dep_simulators[strength[i]] = sim.add_overrotation(single_sys= gate_dep_strengths[i], match=match_clifford).add_overrotation(single_sys=t_gate_rotation, match= match_t).add_overrotation(single_sys=h_gate_rotation, match= match_h).add_overrotation(multi_sys=cnot_gate_rotation, match= match_cnot)
        print(f"Gate independent noise model infidelity with strength {strength[i]}:")
        print(1 - sum([process_fidelity(circ, gate_dep_simulators[strength[i]]) for circ in circs]) / len(circs))

    return gate_dep_simulators

gate_dependent_simulators()

Gate independent noise model infidelity with strength 1:
6.250479676050968e-05
Gate independent noise model infidelity with strength 2:
0.00024999800616931633
Gate independent noise model infidelity with strength 3:
0.0005624160931698707
Gate independent noise model infidelity with strength 4:
0.0009996531912048745


{1: <trueq.simulation.simulator.Simulator at 0x189ae60c0>,
 2: <trueq.simulation.simulator.Simulator at 0x189a86bd0>,
 3: <trueq.simulation.simulator.Simulator at 0x189ad2a20>,
 4: <trueq.simulation.simulator.Simulator at 0x189ad43b0>}

In [2]:
x_strength = 0.014236438
x_rotation_strengths = [x_strength*0.5, x_strength, x_strength*1.5, x_strength*2]

def zxzxz_simulators():

    circs = [ZXZXZ_decompose(tq.Circuit([{0:easy}])) for easy in easy_gates]
    zxzxz_simulators = {}
    for i in range(len(x_rotation_strengths)):

        sim = tq.Simulator()
        
        zxzxz_simulators[strength[i]] = sim.add_overrotation(single_sys= x_rotation_strengths[i], match=tqs.GateMatch(tq.Gate.sx)).add_overrotation(single_sys=t_gate_rotation, match= match_t).add_overrotation(single_sys=h_gate_rotation, match= match_h).add_overrotation(multi_sys=cnot_gate_rotation, match= match_cnot)
        print(f"ZXZXZ decomposition noise model infidelity with strength {strength[i]}:")
        print(1 - sum([process_fidelity(circ, zxzxz_simulators[strength[i]]) for circ in circs]) / len(circs))

    return zxzxz_simulators

In [3]:
zxzxz_simulators()

ZXZXZ decomposition noise model infidelity with strength 1:
6.250781966787855e-05
ZXZXZ decomposition noise model infidelity with strength 2:
0.0002500000208510711
ZXZXZ decomposition noise model infidelity with strength 3:
0.0005623828457183011
ZXZXZ decomposition noise model infidelity with strength 4:
0.000999500083317506


{1: <trueq.simulation.simulator.Simulator at 0x189447f20>,
 2: <trueq.simulation.simulator.Simulator at 0x1897092b0>,
 3: <trueq.simulation.simulator.Simulator at 0x18938f8f0>,
 4: <trueq.simulation.simulator.Simulator at 0x18973a660>}

In [4]:
def rotation_about_axis(theta: float, n: np.ndarray) -> np.ndarray:
    """
    Return the 2x2 unitary V = exp(-i * theta/2 * (n · σ)),
    where n is a 3D unit vector and σ = (X, Y, Z).
    """
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)  # ensure unit length

    nx, ny, nz = n

    H = nx * Gate.x.mat + ny * Gate.y.mat + nz * Gate.z.mat  # generator n · σ

    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)

    V = c * Gate.i.mat - 1j * s * H
    return V

phi = 0.08  ## RAD, which is approximately 4.58 degrees tilted off the x-axis to the y plane       
n = np.array([np.cos(phi), np.sin(phi), 0.0])
n = n / np.linalg.norm(n)

In [5]:
x_strength_tilted = 0.02236255
x_rotation_strengths_tilted = [x_strength_tilted*0.5, x_strength_tilted, x_strength_tilted*1.5, x_strength_tilted*2]

def zxzxz_simulators_tilted():

    zxzxz_simulators = {}
    for i in range(len(x_rotation_strengths_tilted)):

        V = rotation_about_axis(x_rotation_strengths_tilted[i], n)
        circs = [ZXZXZ_decompose(tq.Circuit([{0:easy}])) for easy in easy_gates]

        sim = tq.Simulator()
        easy_dict = {gate: tq.Gate(V) @ gate.mat for gate in clifford_gates}

        def gate_replace(gate):
            return easy_dict[gate]
        
        zxzxz_simulators[strength[i]] = sim.add_gate_replace(gate_replace, match=tqs.GateMatch(tq.Gate.sx)).add_overrotation(single_sys=t_gate_rotation, match= match_t).add_overrotation(single_sys=h_gate_rotation, match= match_h).add_overrotation(multi_sys=cnot_gate_rotation, match= match_cnot)
        print(f"ZXZXZ decomposition noise model infidelity with strength {strength[i]}:")
        print(1 - sum([process_fidelity(circ, zxzxz_simulators[strength[i]]) for circ in circs]) / len(circs))

    return zxzxz_simulators

zxzxz_simulators_tilted()

ZXZXZ decomposition noise model infidelity with strength 1:
6.250786275585618e-05
ZXZXZ decomposition noise model infidelity with strength 2:
0.00025000034240807434
ZXZXZ decomposition noise model infidelity with strength 3:
0.0005623841286529485
ZXZXZ decomposition noise model infidelity with strength 4:
0.0009995037561006948


{1: <trueq.simulation.simulator.Simulator at 0x18973bda0>,
 2: <trueq.simulation.simulator.Simulator at 0x18938d520>,
 3: <trueq.simulation.simulator.Simulator at 0x189447e00>,
 4: <trueq.simulation.simulator.Simulator at 0x189447c80>}

In [6]:
def gate_independent_simulators():
    gate_ind_simulators = {}

    # Strength 1
    sim1 = tq.Simulator()
    easy_dict_1 = {gate: tq.Gate.rp("X", 1.81215 * 20) @ gate.mat for gate in clifford_gates}
    def gate_replace_1(gate):
        return easy_dict_1[gate]
    noisy_sim_1 = (
        sim1.add_gate_replace(gate_replace_1, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_1 = 1 - sum(process_fidelity(circ, noisy_sim_1) for circ in circs) / len(circs)
    gate_ind_simulators[1] = (noisy_sim_1, infid_1)

    # Strength 2
    sim2 = tq.Simulator()
    easy_dict_2 = {gate: tq.Gate.rp("X", 1.81215) @ gate.mat for gate in clifford_gates}
    def gate_replace_2(gate):
        return easy_dict_2[gate]
    noisy_sim_2 = (
        sim2.add_gate_replace(gate_replace_2, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_2 = 1 - sum(process_fidelity(circ, noisy_sim_2) for circ in circs) / len(circs)
    gate_ind_simulators[2] = (noisy_sim_2, infid_2)

    # Strength 3
    sim3 = tq.Simulator()
    easy_dict_3 = {gate: tq.Gate.rp("X", 1.81215 * 6) @ gate.mat for gate in clifford_gates}
    def gate_replace_3(gate):
        return easy_dict_3[gate]
    noisy_sim_3 = (
        sim3.add_gate_replace(gate_replace_3, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_3 = 1 - sum(process_fidelity(circ, noisy_sim_3) for circ in circs) / len(circs)
    gate_ind_simulators[3] = (noisy_sim_3, infid_3)

    # Strength 4
    sim4 = tq.Simulator()
    easy_dict_4 = {gate: tq.Gate.rp("X", 1.81215 * 12) @ gate.mat for gate in clifford_gates}
    def gate_replace_4(gate):
        return easy_dict_4[gate]
    noisy_sim_4 = (
        sim4.add_gate_replace(gate_replace_4, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_4 = 1 - sum(process_fidelity(circ, noisy_sim_4) for circ in circs) / len(circs)
    gate_ind_simulators[4] = (noisy_sim_4, infid_4)

    return gate_ind_simulators

In [7]:
gate_independent_simulators()[4]

(<trueq.simulation.simulator.Simulator at 0x18976a750>, 0.03558164317233836)

In [30]:
strength_scales = [0.5,1,1.5,2]
x_rotation_strengths = [0.014236438 * scale for scale in strength_scales]

zxzxz_dict = {gate: ZXZXZ_decompose(tq.Circuit([{0:gate}])) for gate in clifford_gates}

circs = [ZXZXZ_decompose(tq.Circuit([{0:easy}])) for easy in easy_gates]
zxzxz_simulators = {}
noisy_easy_gates = {}

for label, x_strength in zip(strength, x_rotation_strengths):

    sim = tq.Simulator()
    
    noisy_sim = (
        sim.add_overrotation(single_sys=x_strength, match=tqs.GateMatch(tq.Gate.sx))
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infidelity = 1 - sum([process_fidelity(circ, noisy_sim) for circ in circs]) / len(circs)

    zxzxz_simulators[label] = (noisy_sim, infidelity)
    noisy_easy_gates[label] = {gate: zxzxz_simulators[label][0].operator(zxzxz_dict[gate]).mat() for gate in clifford_gates}

def gate_replace_1(gate):
    return noisy_easy_gates[1][gate]

def gate_replace_2(gate):
    return noisy_easy_gates[2][gate]

def gate_replace_3(gate):
    return noisy_easy_gates[3][gate]

def gate_replace_4(gate):
    return noisy_easy_gates[4][gate]

sim1 = tq.Simulator()
noisy_sim_1 = (
        sim1.add_gate_replace(gate_replace_1, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )

sim2 = tq.Simulator()
noisy_sim_2 = (
        sim2.add_gate_replace(gate_replace_2, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )

sim3 = tq.Simulator()
noisy_sim_3 = (
        sim3.add_gate_replace(gate_replace_3, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )

sim4 = tq.Simulator()
noisy_sim_4 = (
        sim4.add_gate_replace(gate_replace_4, match=match_clifford)
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )

final_sims = {}
zxzxz_simulators_final = {}

final_sims[1] = noisy_sim_1
final_sims[2] = noisy_sim_2
final_sims[3] = noisy_sim_3
final_sims[4] = noisy_sim_4

for label in strength:
    zxzxz_simulators_final[label] = (final_sims[label], zxzxz_simulators[label][1])

zxzxz_simulators_final


{1: (<trueq.simulation.simulator.Simulator at 0x184e4a6c0>,
  6.250781966787855e-05),
 2: (<trueq.simulation.simulator.Simulator at 0x18997e4e0>,
  0.0002500000208510711),
 3: (<trueq.simulation.simulator.Simulator at 0x189982b40>,
  0.0005623828457183011),
 4: (<trueq.simulation.simulator.Simulator at 0x189982ae0>,
  0.000999500083317506)}

In [34]:

zxzxz_dict = {gate: ZXZXZ_decompose(tq.Circuit([{0:gate}])) for gate in clifford_gates}
zxzxz_simulators = {}
circs = [ZXZXZ_decompose(tq.Circuit([{0: easy}])) for easy in easy_gates]
noisy_easy_gates = {}

# Strength 1 (0.5 * x_strength_tilted)
sim1 = tq.Simulator()
V1 = rotation_about_axis(x_strength_tilted * 0.5, n)
easy_dict_1 = {gate: tq.Gate(V1 @ gate.mat) for gate in clifford_gates}
def gate_replace_1(gate):
    return easy_dict_1[gate]
noisy_sim_1 = (
    sim1.add_gate_replace(gate_replace_1, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys=t_gate_rotation, match=match_t)
        .add_overrotation(single_sys=h_gate_rotation, match=match_h)
        .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
)
infid_1 = 1 - sum(process_fidelity(circ, noisy_sim_1) for circ in circs) / len(circs)
zxzxz_simulators[1] = (noisy_sim_1, infid_1)

# Strength 2 (1.0 * x_strength_tilted)
sim2 = tq.Simulator()
V2 = rotation_about_axis(x_strength_tilted * 1.0, n)
easy_dict_2 = {gate: tq.Gate(V2) @ gate.mat for gate in clifford_gates}
def gate_replace_2(gate):
    return easy_dict_2[gate]
noisy_sim_2 = (
    sim2.add_gate_replace(gate_replace_2, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys=t_gate_rotation, match=match_t)
        .add_overrotation(single_sys=h_gate_rotation, match=match_h)
        .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
)
infid_2 = 1 - sum(process_fidelity(circ, noisy_sim_2) for circ in circs) / len(circs)
zxzxz_simulators[2] = (noisy_sim_2, infid_2)

# Strength 3 (1.5 * x_strength_tilted)
sim3 = tq.Simulator()
V3 = rotation_about_axis(x_strength_tilted * 1.5, n)
easy_dict_3 = {gate: tq.Gate(V3) @ gate.mat for gate in clifford_gates}
def gate_replace_3(gate):
    return easy_dict_3[gate]
noisy_sim_3 = (
    sim3.add_gate_replace(gate_replace_3, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys=t_gate_rotation, match=match_t)
        .add_overrotation(single_sys=h_gate_rotation, match=match_h)
        .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
)
infid_3 = 1 - sum(process_fidelity(circ, noisy_sim_3) for circ in circs) / len(circs)
zxzxz_simulators[3] = (noisy_sim_3, infid_3)

# Strength 4 (2.0 * x_strength_tilted)
sim4 = tq.Simulator()
V4 = rotation_about_axis(x_strength_tilted * 2.0, n)
easy_dict_4 = {gate: tq.Gate(V4) @ gate.mat for gate in clifford_gates}
def gate_replace_4(gate):
    return easy_dict_4[gate]
noisy_sim_4 = (
    sim4.add_gate_replace(gate_replace_4, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys=t_gate_rotation, match=match_t)
        .add_overrotation(single_sys=h_gate_rotation, match=match_h)
        .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
)
infid_4 = 1 - sum(process_fidelity(circ, noisy_sim_4) for circ in circs) / len(circs)
zxzxz_simulators[4] = (noisy_sim_4, infid_4)

for label in strength:
    noisy_easy_gates[label] = {gate: zxzxz_simulators[label][0].operator(zxzxz_dict[gate]).mat() for gate in clifford_gates}

final_sims = {}
zxzxz_simulators_final = {}

final_sims[1] = noisy_sim_1
final_sims[2] = noisy_sim_2
final_sims[3] = noisy_sim_3
final_sims[4] = noisy_sim_4

for label in strength:
    zxzxz_simulators_final[label] = (final_sims[label], zxzxz_simulators[label][1])

zxzxz_simulators_final

{1: (<trueq.simulation.simulator.Simulator at 0x1891c3d10>,
  6.250786275585618e-05),
 2: (<trueq.simulation.simulator.Simulator at 0x1897b6630>,
  0.00025000034240807434),
 3: (<trueq.simulation.simulator.Simulator at 0x1899832f0>,
  0.0005623841286529485),
 4: (<trueq.simulation.simulator.Simulator at 0x189982060>,
  0.0009995037561006948)}

In [38]:
easy_dict_1[Gate.z]

Gate(Y, X, ...)

In [33]:
zxzxz_simulators_final[3][0].operator(circuit = tq.Circuit([{0: Gate.y}])).mat()

array([[0.+0.j, 0.-1.j],
       [0.+1.j, 0.+0.j]])

In [39]:
x_rotation_strengths = [0.014236438 * scale for scale in strength_scales]

def zxzxz_simulators():

    zxzxz_dict = {gate: ZXZXZ_decompose(tq.Circuit([{0:gate}])) for gate in clifford_gates}

    circs = [ZXZXZ_decompose(tq.Circuit([{0:easy}])) for easy in easy_gates]
    zxzxz_simulators = {}
    noisy_easy_gates = {}

    for label, x_strength in zip(strength, x_rotation_strengths):

        sim = tq.Simulator()
        
        noisy_sim = (
            sim.add_overrotation(single_sys=x_strength, match=tqs.GateMatch(tq.Gate.sx))
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )
        infidelity = 1 - sum([process_fidelity(circ, noisy_sim) for circ in circs]) / len(circs)

        zxzxz_simulators[label] = (noisy_sim, infidelity)
        noisy_easy_gates[label] = {gate: zxzxz_simulators[label][0].operator(zxzxz_dict[gate]).mat() for gate in clifford_gates}

    def gate_replace_1(gate):
        return tq.Gate(noisy_easy_gates[1][gate])

    def gate_replace_2(gate):
        return tq.Gate(noisy_easy_gates[2][gate])

    def gate_replace_3(gate):
        return tq.Gate(noisy_easy_gates[3][gate])

    def gate_replace_4(gate):
        return tq.Gate(noisy_easy_gates[4][gate])

    sim1 = tq.Simulator()
    noisy_sim_1 = (
            sim1.add_gate_replace(gate_replace_1, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim2 = tq.Simulator()
    noisy_sim_2 = (
            sim2.add_gate_replace(gate_replace_2, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim3 = tq.Simulator()
    noisy_sim_3 = (
            sim3.add_gate_replace(gate_replace_3, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim4 = tq.Simulator()
    noisy_sim_4 = (
            sim4.add_gate_replace(gate_replace_4, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    final_sims = {}
    zxzxz_simulators_final = {}

    final_sims[1] = noisy_sim_1
    final_sims[2] = noisy_sim_2
    final_sims[3] = noisy_sim_3
    final_sims[4] = noisy_sim_4

    for label in strength:
        zxzxz_simulators_final[label] = (final_sims[label], zxzxz_simulators[label][1])

    return zxzxz_simulators_final

sim_ZXZXZ = zxzxz_simulators()

In [51]:
################ Twisted ZXZXZ Decomposition #####################

def rotation_about_axis(theta: float, n: np.ndarray) -> np.ndarray:
    """
    Return the 2x2 unitary V = exp(-i * theta/2 * (n · σ)),
    where n is a 3D unit vector and σ = (X, Y, Z).
    """
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)  # ensure unit length

    nx, ny, nz = n

    H = nx * Gate.x.mat + ny * Gate.y.mat + nz * Gate.z.mat  # generator n · σ

    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)

    V = c * Gate.i.mat - 1j * s * H
    return V

phi = 0.08  ## RAD, which is approximately 4.58 degrees tilted off the x-axis to the y plane       
n = np.array([np.cos(phi), np.sin(phi), 0.0])
n = n / np.linalg.norm(n)

x_strength_tilted = 0.02236255
x_rotation_strengths_tilted = [x_strength_tilted * scale for scale in strength_scales]

def zxzxz_simulators_tilted():

    zxzxz_dict = {gate: ZXZXZ_decompose(tq.Circuit([{0:gate}])) for gate in clifford_gates}
    zxzxz_simulators = {}
    circs = [ZXZXZ_decompose(tq.Circuit([{0: easy}])) for easy in easy_gates]
    noisy_easy_gates = {}

    # Strength 1 (0.5 * x_strength_tilted)
    sim1 = tq.Simulator()
    V1 = rotation_about_axis(x_strength_tilted * 0.5, n)
    easy_dict_1 = {gate: tq.Gate(V1) @ gate.mat for gate in clifford_gates}
    def gate_replace_1(gate):
        return easy_dict_1[gate]
    noisy_sim_1 = (
        sim1.add_gate_replace(gate_replace_1, match=tqs.GateMatch(tq.Gate.sx))
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_1 = 1 - sum(process_fidelity(circ, noisy_sim_1) for circ in circs) / len(circs)
    zxzxz_simulators[1] = (noisy_sim_1, infid_1)

    # Strength 2 (1.0 * x_strength_tilted)
    sim2 = tq.Simulator()
    V2 = rotation_about_axis(x_strength_tilted * 1.0, n)
    easy_dict_2 = {gate: tq.Gate(V2) @ gate.mat for gate in clifford_gates}
    def gate_replace_2(gate):
        return easy_dict_2[gate]
    noisy_sim_2 = (
        sim2.add_gate_replace(gate_replace_2, match=tqs.GateMatch(tq.Gate.sx))
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_2 = 1 - sum(process_fidelity(circ, noisy_sim_2) for circ in circs) / len(circs)
    zxzxz_simulators[2] = (noisy_sim_2, infid_2)

    # Strength 3 (1.5 * x_strength_tilted)
    sim3 = tq.Simulator()
    V3 = rotation_about_axis(x_strength_tilted * 1.5, n)
    easy_dict_3 = {gate: tq.Gate(V3) @ gate.mat for gate in clifford_gates}
    def gate_replace_3(gate):
        return easy_dict_3[gate]
    noisy_sim_3 = (
        sim3.add_gate_replace(gate_replace_3, match=tqs.GateMatch(tq.Gate.sx))
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_3 = 1 - sum(process_fidelity(circ, noisy_sim_3) for circ in circs) / len(circs)
    zxzxz_simulators[3] = (noisy_sim_3, infid_3)

    # Strength 4 (2.0 * x_strength_tilted)
    sim4 = tq.Simulator()
    V4 = rotation_about_axis(x_strength_tilted * 2.0, n)
    easy_dict_4 = {gate: tq.Gate(V4) @ gate.mat for gate in clifford_gates}
    def gate_replace_4(gate):
        return easy_dict_4[gate]
    noisy_sim_4 = (
        sim4.add_gate_replace(gate_replace_4, match=tqs.GateMatch(tq.Gate.sx))
            .add_overrotation(single_sys=t_gate_rotation, match=match_t)
            .add_overrotation(single_sys=h_gate_rotation, match=match_h)
            .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
    )
    infid_4 = 1 - sum(process_fidelity(circ, noisy_sim_4) for circ in circs) / len(circs)
    zxzxz_simulators[4] = (noisy_sim_4, infid_4)

    for label in strength:
        noisy_easy_gates[label] = {gate: zxzxz_simulators[label][0].operator(zxzxz_dict[gate]).mat() for gate in clifford_gates}

    def gate_replace_1_tilted(gate):
        return tq.Gate(noisy_easy_gates[1][gate])

    def gate_replace_2_tilted(gate):
        return tq.Gate(noisy_easy_gates[2][gate])

    def gate_replace_3_tilted(gate):
        return tq.Gate(noisy_easy_gates[3][gate])

    def gate_replace_4_tilted(gate):
        return tq.Gate(noisy_easy_gates[4][gate])

    sim1 = tq.Simulator()
    noisy_sim_1 = (
            sim1.add_gate_replace(gate_replace_1_tilted, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim2 = tq.Simulator()
    noisy_sim_2 = (
            sim2.add_gate_replace(gate_replace_2_tilted, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim3 = tq.Simulator()
    noisy_sim_3 = (
            sim3.add_gate_replace(gate_replace_3_tilted, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    sim4 = tq.Simulator()
    noisy_sim_4 = (
            sim4.add_gate_replace(gate_replace_4_tilted, match=match_clifford)
                .add_overrotation(single_sys=t_gate_rotation, match=match_t)
                .add_overrotation(single_sys=h_gate_rotation, match=match_h)
                .add_overrotation(multi_sys=cnot_gate_rotation, match=match_cnot)
        )

    final_sims = {}
    zxzxz_simulators_final = {}

    final_sims[1] = noisy_sim_1
    final_sims[2] = noisy_sim_2
    final_sims[3] = noisy_sim_3
    final_sims[4] = noisy_sim_4

    for label in strength:
        zxzxz_simulators_final[label] = (final_sims[label], zxzxz_simulators[label][1])

    return zxzxz_simulators_final

sim_ZXZXZ_tilted = zxzxz_simulators_tilted()

In [55]:
sim_ZXZXZ_tilted[4][0].operator(circuit = tq.Circuit([{0: Gate.z}])).mat()

array([[-0.00174667+9.99996807e-01j,  0.00182633-3.19318099e-06j],
       [-0.00182633-3.19318099e-06j, -0.00174667-9.99996807e-01j]])

In [2]:
import pandas as pd

df1 = pd.read_csv('simulation_results.csv')
df2 = pd.read_csv('simulation_results_ZXZXZ.csv')
df3 = pd.read_csv('simulation_results_ZXZXZ_2.csv')

In [3]:
df_big = pd.concat([df1, df2, df3], ignore_index=True)
df_big.to_csv('simulation_results_combined.csv', index=False)

In [4]:
df_big.shape

(64, 8)